In [ ]:
# importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
# from sklearn.datasets import make_classification

def show_hist(df, features):
    len_ft = len(features)
    fig, axes = plt.subplots(1, len_ft, figsize=(12,3))
    if isinstance(axes, np.ndarray) is False:
        axes = np.array([axes])
    for idx, ft in enumerate(features):
        sns.histplot(df[ft], kde=True, ax=axes[idx])
        axes[idx].set_title(ft)
    plt.tight_layout()
    plt.show()

In [ ]:
main_df = pd.read_csv('diabetes.csv')

print("Shape:", main_df.shape)
print(main_df.columns.tolist())
print()

label = "class"
select_features = [
    "Age",
    "Gender",
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "Genital thrush",
    "visual blurring",
    "Itching",
    "Irritability",
    "delayed healing",
    "partial paresis",
    "muscle stiffness",
    "Alopecia",
    "Obesity",
]
select_columns = select_features + [label]
main_df = main_df[select_columns]
print("Shape:", main_df.shape)
print(main_df.columns.tolist())
print()

print(pd.concat([main_df[:5], main_df[-5:]]))
print()

In [ ]:
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder
# categorical_features = ['Gender']
# preprocess = ColumnTransformer(
#     transformers=[('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)],
#     remainder='passthrough'   # keep the rest of the columns (e.g., 'sales')
# )

pd.set_option('future.no_silent_downcasting', True)

prep_columns = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "Genital thrush",
    "visual blurring",
    "Itching",
    "Irritability",
    "delayed healing",
    "partial paresis",
    "muscle stiffness",
    "Alopecia",
    "Obesity",
]

main_df['Gender'] = main_df['Gender'].replace({'Male': 0, 'Female': 1})
main_df['class'] = main_df['class'].replace({'Positive': 1, 'Negative': 0})
for prep_col in prep_columns:
    main_df[prep_col] = main_df[prep_col].replace({'Yes': 0, 'No': 1})

main_df['Gender'] = pd.to_numeric(main_df['Gender'], errors='coerce')
main_df['class'] = pd.to_numeric(main_df['class'], errors='coerce')
for prep_col in prep_columns:
    main_df[prep_col] = pd.to_numeric(main_df[prep_col], errors='coerce')

print("Shape:", main_df.shape)
print(pd.concat([main_df[:5], main_df[-5:]]))

In [ ]:
show_hist(df=main_df, features=select_columns[:5])
show_hist(df=main_df, features=select_columns[5:11])
show_hist(df=main_df, features=select_columns[11:])

In [ ]:
# cols = prep_columns
# for col in cols:
#     scaler = StandardScaler()
#     main_df[col] = scaler.fit_transform(main_df[[col]].copy())
# print(pd.concat([main_df[:5], main_df[-5:]]))
# show_hist(df=main_df, features=cols)

In [ ]:
# select_features = []
X = main_df[select_features].to_numpy()
y = main_df[[label]].to_numpy().reshape(-1)
print(X.shape)
print(X[:5])
print(y.shape)
print(y[:10])

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

In [ ]:
print(f"Length: {len(X_train)}")
print(X_train[:5])
print(y_train[:5])

In [ ]:
print(f"Length: {len(X_test)}")
print(X_test[:5])
print(y_test[:5])

In [ ]:
from tensorflow.keras import Sequential, layers, optimizers, losses

model = Sequential([
    layers.Dense(15, activation='tanh', input_shape=(len(select_features),)),
    layers.Dense(12, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# model = Sequential([
#     layers.Dense(12, activation='tanh', input_shape=(len(select_features),)),
#     layers.Dense(8, activation='relu'),
#     layers.Dense(1, activation='sigmoid')
# ])

# model = Sequential([
#     layers.Dense(8, activation='tanh', input_shape=(len(select_features),)),
#     layers.Dense(1, activation='sigmoid')
# ])

# model = Sequential([
#     layers.Dense(1, activation='sigmoid', input_shape=(len(select_features),)),
# ])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0005),
    loss=losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=150,
    batch_size=30,
    validation_split=0.1,
    verbose=2
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.2f}")

loss, accuracy = model.evaluate(X_train, y_train, verbose=0)
print(f"Train Accuracy: {accuracy:.2f}")

In [ ]:
prediction = model.predict(X_test[:10])
print(prediction)

In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_pred = np.round(y_pred)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

# Plot the confusion matrix using Seaborn
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 6))
# target_names = ["Yes", "No"]
# sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=target_names, yticklabels=target_names)
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='blue')
plt.plot(history.history['val_accuracy'],
         label='Validation Accuracy', color='orange')
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)

plt.suptitle("Model Training Performance", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
from eli5.permutation_importance import get_score_importances

def predict(X):
    pred = model.predict(X).reshape(-1)
    pred = np.round(pred)
    return pred

def accuracy(y_true, y_p):
    return np.mean(y_true == y_p)

def score(X, y):
    y_pred = predict(X)
    return accuracy(y, y_pred)

base_score, score_decreases = get_score_importances(score, X, y)
feature_importances = np.mean(score_decreases, axis=0)

In [ ]:
report = np.column_stack([select_features, feature_importances])
report = np.array(sorted(report, key=lambda r: r[1], reverse=True))
print(report.shape)
print(report)

In [ ]:
# del(history)
# del(model)